#MIXX​ – Food Demand Prediction & Waste Intelligence

###Validate the dataset

In [ ]:
import pandas as pd

# 1) Load the CSV (after you upload it to Colab files)
df = pd.read_csv("/mixx_synthetic_restaurant_data.csv")

# 2) Parse date and sort (important for time-based forecasting)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["dish_name", "date"]).reset_index(drop=True)

# 3) Quick inspection
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
display(df.head(10))

# 4) Data quality checks (must be OK before modeling)
print("\nMissing values per column:")
print(df.isna().sum())

print("\nDate range:", df["date"].min(), "to", df["date"].max())
print("Unique dishes:", df["dish_name"].nunique())
print("Unique categories:", df["category"].nunique())

# 5) Define the prediction target (what we will forecast next day)
TARGET = "sold_qty"
print("\nTarget column:", TARGET)
print("Target stats:")
display(df[TARGET].describe())

Shape: (1305, 10)

Columns: ['date', 'weekday', 'dish_name', 'category', 'price_level', 'temperature_c', 'is_holiday', 'cooked_qty', 'sold_qty', 'wasted_qty']


,date,weekday,dish_name,category,price_level,temperature_c,is_holiday,cooked_qty,sold_qty,wasted_qty
0,2025-01-01,Wednesday,Beef Stew,Main,High,17.0,0,98,93,5
1,2025-01-02,Thursday,Beef Stew,Main,High,7.3,0,110,101,9
2,2025-01-03,Friday,Beef Stew,Main,High,26.7,0,115,104,11
3,2025-01-06,Monday,Beef Stew,Main,High,11.9,0,126,117,9
4,2025-01-07,Tuesday,Beef Stew,Main,High,19.4,0,107,99,8
5,2025-01-08,Wednesday,Beef Stew,Main,High,15.2,0,93,89,4
6,2025-01-09,Thursday,Beef Stew,Main,High,3.6,0,108,100,8
7,2025-01-10,Friday,Beef Stew,Main,High,15.6,0,125,114,11
8,2025-01-13,Monday,Beef Stew,Main,High,9.8,0,116,102,14
9,2025-01-14,Tuesday,Beef Stew,Main,High,13.0,0,100,94,6



Missing values per column:
date             0
weekday          0
dish_name        0
category         0
price_level      0
temperature_c    0
is_holiday       0
cooked_qty       0
sold_qty         0
wasted_qty       0
dtype: int64

Date range: 2025-01-01 00:00:00 to 2025-12-31 00:00:00
Unique dishes: 5
Unique categories: 4

Target column: sold_qty
Target stats:


,sold_qty
count,1305.000000
mean,106.986207
std,28.901405
min,42.000000
25%,85.000000
50%,103.000000
75%,128.000000
max,194.000000


##Model dataset shape

In [ ]:
import numpy as np

# Make sure sorted correctly (very important)
df = df.sort_values(["dish_name", "date"]).reset_index(drop=True)

# Create next-day target per dish
df["target_next_day"] = df.groupby("dish_name")["sold_qty"].shift(-1)

#Create lag features (what model can use to predict tomorrow)

# Yesterday demand
df["lag_1"] = df.groupby("dish_name")["sold_qty"].shift(1)

# Demand 7 days ago (weekly pattern)
df["lag_7"] = df.groupby("dish_name")["sold_qty"].shift(7)

# 7-day rolling average (trend)
df["rolling_mean_7"] = (
    df.groupby("dish_name")["sold_qty"]
    .shift(1)
    .rolling(7)
    .mean()
)

#Remove rows where lag/target is NaN
df_model = df.dropna().reset_index(drop=True)

print("Model dataset shape:", df_model.shape)
display(df_model.head(10))

Model dataset shape: (1265, 14)


,date,weekday,dish_name,category,price_level,temperature_c,is_holiday,cooked_qty,sold_qty,wasted_qty,target_next_day,lag_1,lag_7,rolling_mean_7
0,2025-01-10,Friday,Beef Stew,Main,High,15.6,0,125,114,11,102.0,100.0,93.0,100.428571
1,2025-01-13,Monday,Beef Stew,Main,High,9.8,0,116,102,14,94.0,114.0,101.0,103.428571
2,2025-01-14,Tuesday,Beef Stew,Main,High,13.0,0,100,94,6,98.0,102.0,104.0,103.571429
3,2025-01-15,Wednesday,Beef Stew,Main,High,-2.2,0,107,98,9,105.0,94.0,117.0,102.142857
4,2025-01-16,Thursday,Beef Stew,Main,High,-5.0,0,110,105,5,127.0,98.0,99.0,99.428571
5,2025-01-17,Friday,Beef Stew,Main,High,23.4,0,141,127,14,107.0,105.0,89.0,100.285714
6,2025-01-20,Monday,Beef Stew,Main,High,-3.5,0,122,107,15,87.0,127.0,100.0,105.714286
7,2025-01-21,Tuesday,Beef Stew,Main,High,20.1,0,92,87,5,89.0,107.0,114.0,106.714286
8,2025-01-22,Wednesday,Beef Stew,Main,High,17.2,0,94,89,5,87.0,87.0,102.0,102.857143
9,2025-01-23,Thursday,Beef Stew,Main,High,30.0,0,94,87,7,114.0,89.0,94.0,101.000000


###Time-Based Train/Test Split

In [ ]:
split_date = "2025-11-01"

train = df_model[df_model["date"] < split_date]
test  = df_model[df_model["date"] >= split_date]

print("Train shape:", train.shape)
print("Test shape:", test.shape)

print("\nTrain date range:", train["date"].min(), "to", train["date"].max())
print("Test date range:", test["date"].min(), "to", test["date"].max())

Train shape: (1055, 14)
Test shape: (210, 14)

Train date range: 2025-01-10 00:00:00 to 2025-10-31 00:00:00
Test date range: 2025-11-03 00:00:00 to 2025-12-30 00:00:00


We will:

Train → January to October

Test → November & December

We used a time-based split to simulate real-world forecasting, where the model is trained on historical data and evaluated on unseen future data.

###Define Features & Target

In [ ]:
# Target
y_train = train["target_next_day"]
y_test  = test["target_next_day"]

# Feature columns
feature_cols = [
    "lag_1",
    "lag_7",
    "rolling_mean_7",
    "temperature_c",
    "is_holiday"
]

X_train = train[feature_cols]
X_test  = test[feature_cols]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (1055, 5)
X_test shape: (210, 5)


###Train & Compare Multiple Models

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

def evaluate_model(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    return {
        "Model": name,
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "MAPE (%)": round(mape, 2)
    }

results = []

# Naive Baseline
# Predict tomorrow = today (lag_1)
baseline_pred = X_test["lag_1"]
results.append(evaluate_model("Naive Baseline", y_test, baseline_pred))

# Linear Regression

lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
results.append(evaluate_model("Linear Regression", y_test, lr_pred))


# Random Forest

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
results.append(evaluate_model("Random Forest", y_test, rf_pred))


# Show comparison table

results_df = pd.DataFrame(results)
display(results_df.sort_values("RMSE"))

,Model,MAE,RMSE,MAPE (%)
1,Linear Regression,10.20,13.06,9.36
2,Random Forest,10.29,13.60,9.22
0,Naive Baseline,15.69,19.32,14.69


We compared a naive baseline, a linear model, and a nonlinear ensemble model.
Linear Regression achieved the lowest Root Mean Squared Error (RMSE), indicating that demand patterns in our synthetic restaurant data are largely linear and driven by historical lag features. Therefore, we selected Linear Regression as the final production model.

We predict next-day sold_qty.

RMSE measures the typical prediction error in the same unit as sold_qty (here: “portions/units sold”).

Lower RMSE = better model.

RMSE is useful because a few very wrong predictions matter a lot in restaurant planning (they cause waste or shortage), and RMSE highlights that.

###Daily Retrain + Predict Next Day

In [ ]:
from sklearn.linear_model import LinearRegression
import pandas as pd

def mixx_daily_prediction_pipeline(df):

    df = df.sort_values(["dish_name", "date"]).reset_index(drop=True)

    # Create next-day target
    df["target_next_day"] = df.groupby("dish_name")["sold_qty"].shift(-1)

    # Lag features
    df["lag_1"] = df.groupby("dish_name")["sold_qty"].shift(1)
    df["lag_7"] = df.groupby("dish_name")["sold_qty"].shift(7)

    df["rolling_mean_7"] = (
        df.groupby("dish_name")["sold_qty"]
        .shift(1)
        .rolling(7)
        .mean()
    )

    df_model = df.dropna().reset_index(drop=True)

    feature_cols = [
        "lag_1",
        "lag_7",
        "rolling_mean_7",
        "temperature_c",
        "is_holiday"
    ]

    X = df_model[feature_cols]
    y = df_model["target_next_day"]

    # Train on all except last row
    model = LinearRegression()
    model.fit(X[:-1], y[:-1])

    # Use last row properly as DataFrame
    X_next = df_model.iloc[[-1]][feature_cols]

    next_day_prediction = model.predict(X_next)[0]

    return round(next_day_prediction, 2)

In [ ]:
next_prediction = mixx_daily_prediction_pipeline(df)
print("Predicted Next Day Demand:", next_prediction)

Predicted Next Day Demand: 91.7


We implemented a rolling retraining system where the model updates automatically when new daily data is added.

###Predict Next Day for All Dishes

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np
import pandas as pd

def predict_next_day_all_dishes_smart(df):

    df = df.sort_values(["dish_name", "date"]).reset_index(drop=True)

    predictions = []

    for dish in df["dish_name"].unique():

        df_dish = df[df["dish_name"] == dish].copy()

        # If not enough history → fallback
        if len(df_dish) < 14:
            fallback_prediction = df_dish["sold_qty"].mean()
            predictions.append({
                "Dish Name": dish,
                "Predicted Next Day Demand": round(fallback_prediction, 2),
                "Model Used": "Fallback Average"
            })
            continue

        # Create features
        df_dish["target_next_day"] = df_dish["sold_qty"].shift(-1)
        df_dish["lag_1"] = df_dish["sold_qty"].shift(1)
        df_dish["lag_7"] = df_dish["sold_qty"].shift(7)

        df_dish["rolling_mean_7"] = (
            df_dish["sold_qty"]
            .shift(1)
            .rolling(7)
            .mean()
        )

        df_model = df_dish.dropna().reset_index(drop=True)

        feature_cols = [
            "lag_1",
            "lag_7",
            "rolling_mean_7",
            "temperature_c",
            "is_holiday"
        ]

        X = df_model[feature_cols]
        y = df_model["target_next_day"]

        model = LinearRegression()
        model.fit(X[:-1], y[:-1])

        X_next = df_model.iloc[[-1]][feature_cols]
        next_day_prediction = model.predict(X_next)[0]

        predictions.append({
            "Dish Name": dish,
            "Predicted Next Day Demand": round(next_day_prediction, 2),
            "Model Used": "Linear Regression"
        })

    return pd.DataFrame(predictions)

In [ ]:
dashboard_df = predict_next_day_all_dishes_smart(df)
display(dashboard_df.sort_values("Predicted Next Day Demand", ascending=False))

,Dish Name,Predicted Next Day Demand,Model Used
3,Rice,138.48,Linear Regression
1,Chicken Curry,122.73,Linear Regression
0,Beef Stew,97.74,Linear Regression
4,Veg Pasta,90.95,Linear Regression
2,Chocolate Pudding,71.16,Linear Regression


We trained an independent forecasting model per dish, allowing dish-level demand planning rather than global demand estimation.

The model predicts next-day demand in terms of number of units sold (servings), based on historical sales patterns.

###Holiday Intelligence

In [ ]:
import holidays

# Get years from dataset
data_years = sorted(df["date"].dt.year.unique())

# Also include next year for forecasting (important for demo!)
next_year = max(data_years) + 1

all_years = data_years + [next_year]

# Create Finland holiday calendar for all required years
fi_holidays = holidays.Finland(years=all_years)

print("Holiday years covered:", all_years)

Holiday years covered: [np.int32(2025), np.int32(2026)]


###Weather & Holiday actually affect tomorrow’s prediction (forecast-aware)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

def predict_next_day_all_dishes_with_forecast(df, temp_tomorrow_c: float, date_tomorrow: str):
    """
    Predict next-day demand per dish using:
      - lag features from history
      - TOMORROW's temperature + Finland holiday flag (auto)
    """

    date_tomorrow = pd.to_datetime(date_tomorrow)

    # Finland holiday auto flag for tomorrow (uses the fi_holidays object from Step 9)
    is_holiday_tomorrow = 1 if date_tomorrow in fi_holidays else 0

    df = df.sort_values(["dish_name", "date"]).reset_index(drop=True)

    predictions = []

    for dish in df["dish_name"].unique():
        df_dish = df[df["dish_name"] == dish].copy()

        # Fallback if new dish / too little history
        if len(df_dish) < 14:
            predictions.append({
                "Dish Name": dish,
                "Predicted Next Day Demand": round(df_dish["sold_qty"].mean(), 2),
                "Model Used": "Fallback Average",
                "Temp Tomorrow (C)": temp_tomorrow_c,
                "Holiday Tomorrow": is_holiday_tomorrow
            })
            continue

        # Features + target
        df_dish["target_next_day"] = df_dish["sold_qty"].shift(-1)
        df_dish["lag_1"] = df_dish["sold_qty"].shift(1)
        df_dish["lag_7"] = df_dish["sold_qty"].shift(7)
        df_dish["rolling_mean_7"] = df_dish["sold_qty"].shift(1).rolling(7).mean()

        df_model = df_dish.dropna().reset_index(drop=True)

        feature_cols = ["lag_1", "lag_7", "rolling_mean_7", "temperature_c", "is_holiday"]

        X = df_model[feature_cols]
        y = df_model["target_next_day"]

        model = LinearRegression()
        model.fit(X[:-1], y[:-1])

        # Build tomorrow feature row from last known lags, but override weather/holiday with TOMORROW values
        X_next = df_model.iloc[[-1]][feature_cols].copy()
        X_next["temperature_c"] = temp_tomorrow_c
        X_next["is_holiday"] = is_holiday_tomorrow

        pred = model.predict(X_next)[0]

        predictions.append({
            "Dish Name": dish,
            "Predicted Next Day Demand": round(pred, 2),
            "Model Used": "Linear Regression",
            "Temp Tomorrow (C)": temp_tomorrow_c,
            "Holiday Tomorrow": is_holiday_tomorrow
        })

    return pd.DataFrame(predictions).sort_values("Predicted Next Day Demand", ascending=False)

In [ ]:
tomorrow_date = "2026-01-01"
tomorrow_temp_c = 2.0

dashboard_forecast = predict_next_day_all_dishes_with_forecast(df, tomorrow_temp_c, tomorrow_date)
display(dashboard_forecast)

,Dish Name,Predicted Next Day Demand,Model Used,Temp Tomorrow (C),Holiday Tomorrow
3,Rice,138.77,Linear Regression,2.0,1
1,Chicken Curry,125.94,Linear Regression,2.0,1
0,Beef Stew,98.91,Linear Regression,2.0,1
4,Veg Pasta,89.90,Linear Regression,2.0,1
2,Chocolate Pudding,72.42,Linear Regression,2.0,1


1️⃣ Problem Definition: The goal is to predict next-day dish-level demand to reduce food waste and improve kitchen planning.

2️⃣ Model Design: We implemented a rolling retraining system.
Each dish has its own forecasting model trained using lag features, temperature, and holiday effects.

keywords:

Time-series forecasting

Lag-based features

External regressors (weather + holiday)

3️⃣ Why Weather Matters: Weather influences customer behavior.
For example, colder days may increase demand for warm dishes like soups or rice-based meals.

4️⃣ Why Holiday Matters

Holiday effect is automatically detected using Finland’s national holiday calendar.
On holidays, restaurant demand patterns differ significantly.

system:

Automatically checks if tomorrow is a Finland holiday

Injects that information into the model

That is very strong design.

5️⃣ Interpretation of This Specific Output

Rice → 147
Chicken Curry → 130
Beef Stew → 109

explain:

Because tomorrow is a holiday and temperature is low (2°C), the model predicts higher demand compared to non-holiday average days.

can even compare:

Run with temperature = 15°C

Run with holiday = 0

And show how numbers change.

**“Staff inputs today’s cooked & sold quantity → system updates → retrains → predicts tomorrow again.”**

###Automatic Weather Fetch

In [ ]:
import requests

def get_tomorrow_temperature(lat=60.1699, lon=24.9384):
    """
    Default coordinates = Helsinki, Finland.
    You can change if needed.
    """

    url = (
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={lat}&longitude={lon}"
        f"&daily=temperature_2m_max"
        f"&timezone=Europe/Helsinki"
    )

    response = requests.get(url)
    data = response.json()

    tomorrow_temp = data["daily"]["temperature_2m_max"][1]  # index 1 = tomorrow

    return tomorrow_temp

###Build Staff Daily Input Function

In [ ]:
def add_daily_data_and_predict(df, date_today, temp_today_c, sold_dict, cooked_dict,
                               new_dish_meta=None):
    """
    new_dish_meta example:
    {
      "Salmon Soup": {"category":"Main", "price_level":"High"},
      "Fruit Salad": {"category":"Dessert", "price_level":"Low"}
    }
    """

    date_today = pd.to_datetime(date_today)
    new_rows = []

    if new_dish_meta is None:
        new_dish_meta = {}

    for dish in sold_dict.keys():
        sold = sold_dict[dish]
        cooked = cooked_dict[dish]
        wasted = cooked - sold

        # If dish exists in historical data -> take category/price_level from there
        if dish in df["dish_name"].values:
            meta_row = df[df["dish_name"] == dish].iloc[0]
            category = meta_row["category"]
            price_level = meta_row["price_level"]
        else:
            # New dish -> take metadata from new_dish_meta, else default
            category = new_dish_meta.get(dish, {}).get("category", "Main")
            price_level = new_dish_meta.get(dish, {}).get("price_level", "Medium")

        new_rows.append({
            "date": date_today,
            "weekday": date_today.day_name(),
            "dish_name": dish,
            "category": category,
            "price_level": price_level,
            "temperature_c": temp_today_c,
            "is_holiday": 1 if date_today in fi_holidays else 0,
            "cooked_qty": cooked,
            "sold_qty": sold,
            "wasted_qty": wasted
        })

    df_updated = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)

    # Predict tomorrow (forecast-aware)
    tomorrow_date = (date_today + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    tomorrow_temp = get_tomorrow_temperature()

    dashboard = predict_next_day_all_dishes_smart(df_updated)  # smart handles new dishes safely

    return df_updated, dashboard, tomorrow_date, tomorrow_temp

###DEMO 1: BASELINE FORECAST (Historical Data Only)

In [ ]:
forecast_temp = get_tomorrow_temperature()

historical_dashboard = predict_next_day_all_dishes_smart(df)

print("📊 Forecast Based ONLY on Historical Data")
print("🌡 Forecast temperature:", forecast_temp, "°C")

display(historical_dashboard.sort_values("Predicted Next Day Demand", ascending=False))

📊 Forecast Based ONLY on Historical Data
🌡 Forecast temperature: 2.0 °C


,Dish Name,Predicted Next Day Demand,Model Used
3,Rice,138.48,Linear Regression
1,Chicken Curry,122.73,Linear Regression
0,Beef Stew,97.74,Linear Regression
4,Veg Pasta,90.95,Linear Regression
2,Chocolate Pudding,71.16,Linear Regression


Staff enters today’s actual sales.
Cooked quantity was based on yesterday’s prediction.
The system automatically detects weather forecast.
It retrains the model with new data.
Then it generates updated demand forecast for tomorrow.

###DEMO 2: Staff Daily Input with new dish

In [ ]:
today_date = "2026-01-01"
today_temp = get_tomorrow_temperature()

sold_today = {
    "Rice": 150,
    "Chicken Curry": 128,
    "Beef Stew": 105,
    "Veg Pasta": 95,
    "Chocolate Pudding": 70,
    "Salmon Soup": 40           # ✅ NEW DISH
}

cooked_today = {
    "Rice": 147,
    "Chicken Curry": 130,
    "Beef Stew": 109,
    "Veg Pasta": 98,
    "Chocolate Pudding": 75,
    "Salmon Soup": 45           # cooked a bit more than sold
}

new_dish_meta = {
    "Salmon Soup": {"category": "Main", "price_level": "High"}
}

df, dashboard, tomorrow_date, tomorrow_temp = add_daily_data_and_predict(
    df, today_date, today_temp, sold_today, cooked_today, new_dish_meta
)

print("Tomorrow date:", tomorrow_date, "| Forecast temp:", tomorrow_temp)
display(dashboard.sort_values("Predicted Next Day Demand", ascending=False))

Tomorrow date: 2026-01-02 | Forecast temp: 2.0


,Dish Name,Predicted Next Day Demand,Model Used
3,Rice,144.15,Linear Regression
1,Chicken Curry,119.50,Linear Regression
0,Beef Stew,104.73,Linear Regression
5,Veg Pasta,95.03,Linear Regression
2,Chocolate Pudding,69.40,Linear Regression
4,Salmon Soup,40.00,Fallback Average


**(New dish) = staff inputs sold/cooked, and a new dish appears with Model Used = Fallback Average (because there’s no history for that dish yet, so your system uses a safe default).**

###DEMO 3: Staff Daily Input with new additional dish

In [ ]:
"""
today_date = "2026-03-12"
today_temp = get_tomorrow_temperature()

# Staff daily input
sold_today = {
    "Rice": 155
    "Chicken Curry": 120,
    "Beef Stew": 108,
    "Veg Pasta": 93,
    "Chocolate Pudding": 72,
    "Salmon Soup": 38
}

cooked_today = {
    "Rice": 150,
    "Chicken Curry": 125,
    "Beef Stew": 110,
    "Veg Pasta": 96,
    "Chocolate Pudding": 75,
    "Salmon Soup": 42
}

# NEW ADDITIONAL DISH
new_dish_meta = {
    "Shrimp Pasta": {"category": "Main", "price_level": "High"}
}

# Run smart update + retrain
df, dashboard, tomorrow_date, tomorrow_temp = add_daily_data_and_predict(
    df,
    today_date,
    today_temp,
    sold_today,
    cooked_today,
    new_dish_meta
)

print("Tomorrow date:", tomorrow_date, "| Forecast temp:", tomorrow_temp)

display(
    dashboard.sort_values("Predicted Next Day Demand", ascending=False)
)"""

'\ntoday_date = "2026-03-12"\ntoday_temp = get_tomorrow_temperature()\n\n# Staff daily input\nsold_today = {\n    "Rice": 155,\n    "Chicken Curry": 120,\n    "Beef Stew": 108,\n    "Veg Pasta": 93,\n    "Chocolate Pudding": 72,\n    "Salmon Soup": 38\n}\n\ncooked_today = {\n    "Rice": 150,\n    "Chicken Curry": 125,\n    "Beef Stew": 110,\n    "Veg Pasta": 96,\n    "Chocolate Pudding": 75,\n    "Salmon Soup": 42\n}\n\n# NEW ADDITIONAL DISH\nnew_dish_meta = {\n    "Shrimp Pasta": {"category": "Main", "price_level": "High"}\n}\n\n# Run smart update + retrain\ndf, dashboard, tomorrow_date, tomorrow_temp = add_daily_data_and_predict(\n    df,\n    today_date,\n    today_temp,\n    sold_today,\n    cooked_today,\n    new_dish_meta\n)\n\nprint("Tomorrow date:", tomorrow_date, "| Forecast temp:", tomorrow_temp)\n\ndisplay(\n    dashboard.sort_values("Predicted Next Day Demand", ascending=False)\n)'

##Architecture:

Google Colab (ML Model)
        ↓
Write predictions to BigQuery table
        ↓
Looker Studio reads BigQuery
        ↓
Dashboard auto refresh

The model writes predictions to BigQuery.
Looker Studio reads directly from BigQuery.
When new data is entered and the model retrains, the dashboard updates automatically.

##Bigquery to Looker Studio

###AUTH

In [ ]:
from google.colab import auth
auth.authenticate_user()
print("Auth done")

Auth done


###CREATE DATASET + TABLE

In [ ]:
from google.cloud import bigquery

PROJECT_ID = "mixx-485612"
DATASET_ID = "mixx"
TABLE_ID = "dashboard_predictions"

client = bigquery.Client(project=PROJECT_ID)

# Dataset
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = "EU"
client.create_dataset(dataset_ref, exists_ok=True)
print(f"Dataset ready: {PROJECT_ID}.{DATASET_ID}")

# Table schema for Looker
table_ref = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

schema = [
    bigquery.SchemaField("run_ts", "TIMESTAMP", mode="REQUIRED"),
    bigquery.SchemaField("scenario", "STRING", mode="REQUIRED"),  # demo1/demo2/demo3
    bigquery.SchemaField("forecast_date", "DATE", mode="REQUIRED"),
    bigquery.SchemaField("dish_name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("predicted_next_day_demand", "FLOAT", mode="REQUIRED"),
    bigquery.SchemaField("model_used", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("temp_c", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("is_holiday", "INTEGER", mode="NULLABLE"),
]

table = bigquery.Table(table_ref, schema=schema)
client.create_table(table, exists_ok=True)
print(f"Table ready: {table_ref}")

Dataset ready: mixx-485612.mixx
Table ready: mixx-485612.mixx.dashboard_predictions


###Function to push 3 demo dashboards to BigQuery

In [ ]:
import pandas as pd

def push_dashboard_to_bigquery(
    dashboard_df: pd.DataFrame,
    scenario: str,
    forecast_date: str,
    temp_c: float = None,
    is_holiday: int = None,
    project_id: str = PROJECT_ID,
    dataset_id: str = DATASET_ID,
    table_id: str = TABLE_ID,
    if_exists: str = "append"  # append is best for Looker history
):
    """
    Push a dashboard dataframe (dish-level predictions) into BigQuery for Looker Studio.
    scenario: "demo1_baseline" / "demo2_daily_input" / "demo3_new_dish"
    forecast_date: "YYYY-MM-DD" (tomorrow date)
    """

    # --- normalize column names ---
    df_out = dashboard_df.copy()

    # Find dish column
    if "Dish Name" in df_out.columns:
        df_out.rename(columns={"Dish Name": "dish_name"}, inplace=True)
    elif "dish_name" not in df_out.columns:
        raise ValueError("Can't find dish column. Expected 'Dish Name' or 'dish_name'.")

    # Find prediction column
    if "Predicted Next Day Demand" in df_out.columns:
        df_out.rename(columns={"Predicted Next Day Demand": "predicted_next_day_demand"}, inplace=True)
    elif "predicted_next_day_demand" not in df_out.columns:
        raise ValueError("Can't find prediction column. Expected 'Predicted Next Day Demand'.")

    # Model used optional
    if "Model Used" in df_out.columns:
        df_out.rename(columns={"Model Used": "model_used"}, inplace=True)
    elif "model_used" not in df_out.columns:
        df_out["model_used"] = None

    # --- add metadata fields ---
    df_out["scenario"] = scenario
    df_out["forecast_date"] = pd.to_datetime(forecast_date).date()
    df_out["run_ts"] = pd.Timestamp.utcnow()

    df_out["temp_c"] = temp_c
    df_out["is_holiday"] = is_holiday

    # Keep only the columns that match schema
    df_out = df_out[
        ["run_ts","scenario","forecast_date","dish_name","predicted_next_day_demand","model_used","temp_c","is_holiday"]
    ]

    # --- write to BigQuery ---
    full_table = f"{project_id}.{dataset_id}.{table_id}"

    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_APPEND" if if_exists == "append" else "WRITE_TRUNCATE"
    )

    job = client.load_table_from_dataframe(df_out, full_table, job_config=job_config)
    job.result()

    print(f"Uploaded {len(df_out)} rows to {full_table} | scenario={scenario} | forecast_date={forecast_date}")
    return df_out

Push Demo 1 — baseline (historical only)

In [ ]:
forecast_temp = get_tomorrow_temperature()
forecast_date = "2026-01-02"  # or tomorrow_date variable

push_dashboard_to_bigquery(
    dashboard_df=historical_dashboard,
    scenario="demo1_baseline",
    forecast_date=forecast_date,
    temp_c=forecast_temp,
    is_holiday=None
)

Uploaded 5 rows to mixx-485612.mixx.dashboard_predictions | scenario=demo1_baseline | forecast_date=2026-01-02


,run_ts,scenario,forecast_date,dish_name,predicted_next_day_demand,model_used,temp_c,is_holiday
0,2026-03-02 03:34:27.437710+00:00,demo1_baseline,2026-01-02,Beef Stew,97.74,Linear Regression,2.0,None
1,2026-03-02 03:34:27.437710+00:00,demo1_baseline,2026-01-02,Chicken Curry,122.73,Linear Regression,2.0,None
2,2026-03-02 03:34:27.437710+00:00,demo1_baseline,2026-01-02,Chocolate Pudding,71.16,Linear Regression,2.0,None
3,2026-03-02 03:34:27.437710+00:00,demo1_baseline,2026-01-02,Rice,138.48,Linear Regression,2.0,None
4,2026-03-02 03:34:27.437710+00:00,demo1_baseline,2026-01-02,Veg Pasta,90.95,Linear Regression,2.0,None


Push Demo 2 — daily input (no new dish)

In [ ]:
push_dashboard_to_bigquery_waste_only(
    dashboard_df=dashboard,
    scenario="demo2_daily_input",
    forecast_date=tomorrow_date,
    temp_c=tomorrow_temp,
    is_holiday=None,
    sold_today=sold_today,
    cooked_today=cooked_today
)

Uploaded 6 rows → mixx-485612.mixx.dashboard_predictions | scenario=demo2_daily_input


,dish_name,predicted_next_day_demand,model_used,scenario,forecast_date,run_ts,temp_c,is_holiday,sold_qty,cooked_qty,waste_qty,waste_pct
0,Beef Stew,104.73,Linear Regression,demo2_daily_input,2026-01-02,2026-03-02 04:21:21.954283+00:00,2.0,None,105,109,4,3.669725
1,Chicken Curry,119.50,Linear Regression,demo2_daily_input,2026-01-02,2026-03-02 04:21:21.954283+00:00,2.0,None,128,130,2,1.538462
2,Chocolate Pudding,69.40,Linear Regression,demo2_daily_input,2026-01-02,2026-03-02 04:21:21.954283+00:00,2.0,None,70,75,5,6.666667
3,Rice,144.15,Linear Regression,demo2_daily_input,2026-01-02,2026-03-02 04:21:21.954283+00:00,2.0,None,150,147,0,0.000000
4,Salmon Soup,40.00,Fallback Average,demo2_daily_input,2026-01-02,2026-03-02 04:21:21.954283+00:00,2.0,None,40,45,5,11.111111
5,Veg Pasta,95.03,Linear Regression,demo2_daily_input,2026-01-02,2026-03-02 04:21:21.954283+00:00,2.0,None,95,98,3,3.061224


Push Demo 3 — daily input with new dish

In [ ]:
push_dashboard_to_bigquery_waste_only(
    dashboard_df=dashboard,
    scenario="demo3_new_dish",
    forecast_date=tomorrow_date,
    temp_c=tomorrow_temp,
    is_holiday=None,
    sold_today=sold_today,
    cooked_today=cooked_today
)

Uploaded 6 rows → mixx-485612.mixx.dashboard_predictions | scenario=demo3_new_dish


,dish_name,predicted_next_day_demand,model_used,scenario,forecast_date,run_ts,temp_c,is_holiday,sold_qty,cooked_qty,waste_qty,waste_pct
0,Beef Stew,104.73,Linear Regression,demo3_new_dish,2026-01-02,2026-03-02 04:20:16.187347+00:00,2.0,None,105,109,4,3.669725
1,Chicken Curry,119.50,Linear Regression,demo3_new_dish,2026-01-02,2026-03-02 04:20:16.187347+00:00,2.0,None,128,130,2,1.538462
2,Chocolate Pudding,69.40,Linear Regression,demo3_new_dish,2026-01-02,2026-03-02 04:20:16.187347+00:00,2.0,None,70,75,5,6.666667
3,Rice,144.15,Linear Regression,demo3_new_dish,2026-01-02,2026-03-02 04:20:16.187347+00:00,2.0,None,150,147,0,0.000000
4,Salmon Soup,40.00,Fallback Average,demo3_new_dish,2026-01-02,2026-03-02 04:20:16.187347+00:00,2.0,None,40,45,5,11.111111
5,Veg Pasta,95.03,Linear Regression,demo3_new_dish,2026-01-02,2026-03-02 04:20:16.187347+00:00,2.0,None,95,98,3,3.061224


###Check Waste

In [ ]:
from google.cloud import bigquery

PROJECT_ID = "mixx-485612"
DATASET_ID = "mixx"
TABLE_ID = "dashboard_predictions"

client = bigquery.Client(project=PROJECT_ID)
table = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

alter_sql = f""" ALTER TABLE `{table}` ADD COLUMN IF NOT EXISTS sold_qty INT64, ADD COLUMN IF NOT EXISTS cooked_qty INT64, ADD COLUMN IF NOT EXISTS waste_qty INT64, ADD COLUMN IF NOT EXISTS waste_pct FLOAT64 """

client.query(alter_sql).result()
print("Added waste columns to BigQuery table")

Added waste columns to BigQuery table


In [ ]:
import pandas as pd
from google.cloud import bigquery

def push_dashboard_to_bigquery_waste_only(
    dashboard_df,
    scenario,
    forecast_date,
    temp_c=None,
    is_holiday=None,
    sold_today=None,
    cooked_today=None,
    project_id="mixx-485612",
    dataset_id="mixx",
    table_id="dashboard_predictions"
):
    client = bigquery.Client(project=project_id)
    table_ref = f"{project_id}.{dataset_id}.{table_id}"

    df_out = dashboard_df.copy()

    # Rename dashboard columns → BigQuery format
    df_out = df_out.rename(columns={
        "Dish Name": "dish_name",
        "Predicted Next Day Demand": "predicted_next_day_demand",
        "Model Used": "model_used"
    })

    # Add metadata
    df_out["scenario"] = str(scenario)
    df_out["forecast_date"] = pd.to_datetime(forecast_date).date()
    df_out["run_ts"] = pd.Timestamp.utcnow()
    df_out["temp_c"] = None if temp_c is None else float(temp_c)
    df_out["is_holiday"] = is_holiday

    # Safe defaults
    sold_today = sold_today or {}
    cooked_today = cooked_today or {}

    # Add waste calculation
    df_out["sold_qty"] = df_out["dish_name"].map(sold_today).fillna(0).astype(int)
    df_out["cooked_qty"] = df_out["dish_name"].map(cooked_today).fillna(0).astype(int)

    df_out["waste_qty"] = (
        df_out["cooked_qty"] - df_out["sold_qty"]
    ).clip(lower=0).astype(int)

    df_out["waste_pct"] = df_out.apply(
        lambda r: (r["waste_qty"] / r["cooked_qty"] * 100.0)
        if r["cooked_qty"] > 0 else 0.0,
        axis=1
    ).astype(float)

    # Delete only this scenario (clean demo switching)
    delete_sql = f"""
        DELETE FROM `{table_ref}`
        WHERE scenario = @scenario
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("scenario", "STRING", str(scenario))
        ]
    )
    client.query(delete_sql, job_config=job_config).result()

    # Upload new rows
    load_job = client.load_table_from_dataframe(df_out, table_ref)
    load_job.result()

    print(f"Uploaded {len(df_out)} rows → {table_ref} | scenario={scenario}")

    return df_out